## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

from src.paths import MANUAL_PATH, PROCESSED_PATH
from src.risk import (
    RISK_FREE_RATE,
    annualized_return,
    annualized_volatility,
    max_drawdown,
)
from src.selection import MIN_RETURN_OBSERVATIONS, ROLE_SCORE_MAP

## 2. Paths

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

MANUAL_PATH.mkdir(parents=True, exist_ok=True)

## 3. Load current holdings

In [ ]:
holdings = pd.read_csv(DATA_PATH / "current_holdings_trackable.csv")

holdings[
    [
        "symbol",
        "name",
        "asset_class",
        "yf_ticker",
        "price_currency",
        "actual_weight"
    ]
]

## 4. Asset universe (example seed, only used on first run)

`data/manual/asset_universe_manual.csv` (gitignored) is the central list of holdings + watchlist. Every **current holding** must be in it — later notebooks read these columns:

- `role`, `role_score` → selection scoring (Notebook 05)
- `history_proxy` (optional) → longer return history (Notebooks 03/05)
- `stress_bucket` → stress scenarios and look-through exposure (Notebooks 09/10); valid values in `src/stress.py` (`STRESS_BUCKETS`)
- `sector` (GICS) and optional `country` → single stocks only, look-through exposure (Notebook 10)

In [ ]:
# Generic example universe (holdings + watchlist) — used only to seed data/manual/asset_universe_manual.csv
# on a fresh checkout (that file is gitignored, see data/manual/*, and holds the
# real, curated watchlist). Edit the CSV directly to change your actual universe;
# this cell intentionally contains no real holdings/watchlist data, since it is
# published as part of the notebook source. stress_bucket drives notebooks 09
# and 10; single stocks also need a sector (GICS) and optionally a country.
# source: "Current holding" or "Watchlist" (watchlist assets can become
# candidates in notebook 05).
asset_universe_example_seed = pd.DataFrame([
    {
        "yf_ticker": "VWCE.DE",
        "name": "Vanguard FTSE All-World",
        "asset_class": "FUND",
        "region": "Global",
        "role": "Core equity",
        "source": "Current holding",
        "stress_bucket": "global_equity"
    },
    {
        "yf_ticker": "IEMA.AS",
        "name": "iShares MSCI EM IMI",
        "asset_class": "FUND",
        "region": "Emerging Markets",
        "role": "Equity diversification",
        "source": "Current holding",
        "stress_bucket": "em_equity"
    },
    {
        "yf_ticker": "4GLD.DE",
        "name": "Xetra-Gold",
        "asset_class": "FUND",
        "region": "Global",
        "role": "Crisis hedge",
        "source": "Current holding",
        "stress_bucket": "gold"
    },
    {
        "yf_ticker": "BTC-EUR",
        "name": "Bitcoin",
        "asset_class": "CRYPTO",
        "region": "Global",
        "role": "Alternative asset",
        "source": "Current holding",
        "stress_bucket": "crypto"
    },
    {
        "yf_ticker": "AAPL",
        "name": "Apple",
        "asset_class": "STOCK",
        "region": "US",
        "role": "Single stock",
        "source": "Current holding",
        "stress_bucket": "us_single_stock",
        "sector": "Technology"
    },
    {
        "yf_ticker": "MSFT",
        "name": "Microsoft",
        "asset_class": "STOCK",
        "region": "US",
        "role": "Single stock",
        "source": "Current holding",
        "stress_bucket": "us_single_stock",
        "sector": "Technology"
    },
    {
        "yf_ticker": "JNJ",
        "name": "Johnson & Johnson",
        "asset_class": "STOCK",
        "region": "US",
        "role": "Single stock",
        "source": "Watchlist",
        "stress_bucket": "us_single_stock",
        "sector": "Health Care"
    },
    {
        "yf_ticker": "V",
        "name": "Visa",
        "asset_class": "STOCK",
        "region": "US",
        "role": "Single stock",
        "source": "Watchlist",
        "stress_bucket": "us_single_stock",
        "sector": "Financials"
    },
    {
        "yf_ticker": "KO",
        "name": "Coca-Cola",
        "asset_class": "STOCK",
        "region": "US",
        "role": "Single stock",
        "source": "Watchlist",
        "stress_bucket": "us_single_stock",
        "sector": "Consumer Staples"
    },
    {
        "yf_ticker": "SAP.DE",
        "name": "SAP",
        "asset_class": "STOCK",
        "region": "Europe",
        "role": "Single stock",
        "source": "Watchlist",
        "stress_bucket": "eu_single_stock",
        "sector": "Technology",
        "country": "Germany"
    },
    {
        "yf_ticker": "SIE.DE",
        "name": "Siemens",
        "asset_class": "STOCK",
        "region": "Europe",
        "role": "Single stock",
        "source": "Watchlist",
        "stress_bucket": "eu_single_stock",
        "sector": "Industrials",
        "country": "Germany"
    }
])

asset_universe_example_seed

In [ ]:
MANUAL_UNIVERSE_PATH = MANUAL_PATH / "asset_universe_manual.csv"

# role_score (0-1) = how strategically important the asset's role is in the
# portfolio, used by notebook 05. Kept per asset in the manual CSV; set it
# deliberately for every new asset.
asset_universe_example_seed["role_score"] = (
    asset_universe_example_seed["role"].map(ROLE_SCORE_MAP)
)

if not MANUAL_UNIVERSE_PATH.exists():
    asset_universe_example_seed.to_csv(MANUAL_UNIVERSE_PATH, index=False)
    print(f"Seeded example universe at {MANUAL_UNIVERSE_PATH}")
else:
    print(f"Using existing universe at {MANUAL_UNIVERSE_PATH} (not overwritten)")

## 5. Load Universe

In [ ]:
asset_universe = pd.read_csv(MANUAL_PATH / "asset_universe_manual.csv")

# Optional history_proxy: a ticker tracking the same index/underlying with a
# longer price history (e.g. another provider's ETF on the same index). Used
# only to back-fill returns before the asset's own data starts, so that all
# assets can be compared over one common window (section 9).
if "history_proxy" not in asset_universe.columns:
    asset_universe["history_proxy"] = np.nan

asset_universe

## 6. Download prices

In [ ]:
START_DATE = "2020-01-01"

tickers = asset_universe["yf_ticker"].dropna().unique().tolist()

proxy_tickers = [
    ticker
    for ticker in asset_universe["history_proxy"].dropna().unique()
    if ticker not in tickers
]

tickers = tickers + proxy_tickers

price_data = {}

for ticker in tickers:
    hist = yf.download(
        ticker,
        start=START_DATE,
        auto_adjust=True,
        progress=False
    )

    if hist.empty:
        print(f"No data for {ticker}")
        continue

    close = hist["Close"]

    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]

    price_data[ticker] = close.rename(ticker)

prices_raw = pd.concat(price_data.values(), axis=1)

prices_raw.tail()

In [ ]:
# Price currency is inferred from the yfinance ticker suffix. The pipeline
# only converts USD -> EUR (via usd_to_eur below); any other non-EUR currency
# would otherwise be treated as EUR silently, so we fail loudly instead.
EUR_TICKER_SUFFIXES = (
    ".DE", ".PA", ".AS", ".MI", ".MC", ".BR", ".LS", ".VI", ".F", ".HE", ".IR"
)
NON_EUR_USD_SUFFIXES = {
    ".L": "GBP", ".SW": "CHF", ".ST": "SEK", ".CO": "DKK", ".OL": "NOK",
    ".TO": "CAD", ".AX": "AUD", ".HK": "HKD", ".T": "JPY",
}


def infer_price_currency(ticker):
    if ticker.endswith("-EUR") or ticker.endswith(EUR_TICKER_SUFFIXES):
        return "EUR"
    for suffix, currency in NON_EUR_USD_SUFFIXES.items():
        if ticker.endswith(suffix):
            return currency
    return "USD"


asset_universe["price_currency"] = asset_universe["yf_ticker"].apply(
    infer_price_currency
)

ticker_currency = (
    asset_universe
    .set_index("yf_ticker")["price_currency"]
    .to_dict()
)

for proxy in proxy_tickers:
    ticker_currency[proxy] = infer_price_currency(proxy)

unsupported_currencies = {
    ticker: currency
    for ticker, currency in ticker_currency.items()
    if currency not in {"EUR", "USD"}
}
assert not unsupported_currencies, (
    "Only EUR and USD price currencies are supported "
    f"(USD is converted via usd_to_eur). Add FX handling for: {unsupported_currencies}"
)

ticker_currency

In [ ]:
eurusd = yf.download(
    "EURUSD=X",
    start=START_DATE,
    auto_adjust=True,
    progress=False
)

eurusd_close = eurusd["Close"]

if isinstance(eurusd_close, pd.DataFrame):
    eurusd_close = eurusd_close.iloc[:, 0]

usd_to_eur = 1 / eurusd_close
usd_to_eur.name = "USD_TO_EUR"

usd_to_eur.tail()

In [ ]:
prices_eur = prices_raw.copy()

for ticker in prices_eur.columns:
    if ticker_currency.get(ticker) == "USD":
        prices_eur[ticker] = prices_eur[ticker] * usd_to_eur

prices_eur.tail()

## 7. Returns

In [ ]:
crypto_tickers = [
    ticker for ticker in prices_eur.columns
    if ticker.endswith("-EUR")
]

non_crypto_tickers = [
    ticker for ticker in prices_eur.columns
    if ticker not in crypto_tickers
]

# Trading days are defined by stocks / ETFs
trading_days = prices_eur[non_crypto_tickers].notna().any(axis=1)

portfolio_trading_index = prices_eur.index[trading_days]

prices_eur_trading = prices_eur.loc[portfolio_trading_index].copy()

# Stocks / ETFs: forward-fill missing prices on trading days
prices_eur_trading[non_crypto_tickers] = (
    prices_eur_trading[non_crypto_tickers]
    .ffill()
)

# Crypto: bring crypto prices onto the same trading-day index
# Weekend movements are reflected in the next trading-day return
for ticker in crypto_tickers:
    prices_eur_trading[ticker] = (
        prices_eur[ticker]
        .reindex(portfolio_trading_index)
        .ffill()
    )

returns = prices_eur_trading.pct_change(fill_method=None)

returns = returns.iloc[1:].copy()

returns.tail()

## 8. Asset Universe Overview

In [ ]:
overview_rows = []

# Crypto trades every calendar day, so its raw price series can include an
# in-progress "today" quote (e.g. on a weekend) that no other asset has yet.
# Comparing that against yesterday's close understates the real 1-day move
# and gives crypto a "latest_date" out of sync with the rest of the table.
# Fix: cap crypto at the last date where the wider market actually traded
# (portfolio_trading_index, from section 7) before taking its last 2
# observations. Non-crypto assets keep their own natural per-ticker lag
# (e.g. EU funds updating a day later than US stocks) — that's real data
# lag, not the same bug, so it's left untouched.
market_reference_date = portfolio_trading_index.max()

for ticker in asset_universe["yf_ticker"]:
    if ticker not in prices_eur.columns:
        overview_rows.append({
            "yf_ticker": ticker,
            "latest_date": pd.NaT,
            "current_price_eur": np.nan,
            "one_day_change": np.nan
        })
        continue

    price_series_eur = prices_eur[ticker]

    if ticker in crypto_tickers:
        price_series_eur = price_series_eur.loc[:market_reference_date]

    price_series_eur = price_series_eur.dropna()

    if price_series_eur.shape[0] >= 2:
        latest_date = price_series_eur.index[-1]
        current_price_eur = price_series_eur.iloc[-1]
        previous_price_eur = price_series_eur.iloc[-2]
        one_day_change = current_price_eur / previous_price_eur - 1

    elif price_series_eur.shape[0] == 1:
        latest_date = price_series_eur.index[-1]
        current_price_eur = price_series_eur.iloc[-1]
        one_day_change = np.nan

    else:
        latest_date = pd.NaT
        current_price_eur = np.nan
        one_day_change = np.nan

    overview_rows.append({
        "yf_ticker": ticker,
        "latest_date": latest_date,
        "current_price_eur": current_price_eur,
        "one_day_change": one_day_change
    })

asset_universe_overview_prices = pd.DataFrame(overview_rows)

asset_universe_overview_prices

In [ ]:
asset_universe_overview = asset_universe.merge(
    asset_universe_overview_prices,
    on="yf_ticker",
    how="left"
)

asset_universe_overview = asset_universe_overview[
    [
        "yf_ticker",
        "name",
        "asset_class",
        "region",
        "role",
        "source",
        "latest_date",
        "current_price_eur",
        "one_day_change"
    ]
]

asset_universe_overview

## 9. Asset-Level Metrics

In [ ]:
TRADING_DAYS = 252

# annualized_return, annualized_volatility, max_drawdown now imported
# from src.risk (see setup cell above). These were previously redefined
# here near-identically to notebooks/03_portfolio_risk_performance.ipynb.

In [ ]:
# Step 1 — back-fill each asset's return history with its history_proxy
# (if set) for the dates before the asset's own returns start.
universe_tickers = [
    ticker for ticker in asset_universe["yf_ticker"]
    if ticker in returns.columns
]

history_proxy_map = (
    asset_universe
    .set_index("yf_ticker")["history_proxy"]
    .dropna()
    .to_dict()
)

returns_extended = returns[universe_tickers].copy()

for ticker, proxy in history_proxy_map.items():
    if ticker not in returns_extended.columns or proxy not in returns.columns:
        continue

    own_start = returns[ticker].first_valid_index()
    before_own_start = returns_extended.index < own_start

    returns_extended.loc[before_own_start, ticker] = (
        returns.loc[before_own_start, proxy]
    )

# Step 2 — common comparison window. Every metric below is computed from the
# same start date for all assets, otherwise an asset with a short history
# (e.g. no 2020 crash / 2022 bear market in its window) looks artificially
# good on drawdown/volatility/Sharpe. Assets that are too short even with a
# proxy (< MIN_RETURN_OBSERVATIONS) don't determine the window; they keep
# their own history and are flagged "limited data" in notebook 05.
history_start = {
    ticker: returns_extended[ticker].first_valid_index()
    for ticker in universe_tickers
}

long_enough = [
    ticker for ticker in universe_tickers
    if returns_extended[ticker].dropna().shape[0] >= MIN_RETURN_OBSERVATIONS
]

metrics_window_start = max(history_start[ticker] for ticker in long_enough)

print(f"Common metrics window: {metrics_window_start.date()} to {returns_extended.index.max().date()}")


def metric_returns(ticker):
    series = returns_extended[ticker]
    if ticker in long_enough:
        series = series.loc[metrics_window_start:]
    return series.dropna()


asset_return_stats = pd.DataFrame({
    "yf_ticker": universe_tickers,
    "data_start": [
        prices_eur[ticker].first_valid_index()
        for ticker in universe_tickers
    ],
    "data_end": [
        prices_eur[ticker].last_valid_index()
        for ticker in universe_tickers
    ],
    "uses_history_proxy": [
        ticker in history_proxy_map
        and history_start[ticker] < returns[ticker].first_valid_index()
        for ticker in universe_tickers
    ],
    "metrics_start": [
        metric_returns(ticker).index.min()
        for ticker in universe_tickers
    ],
    "n_return_observations": [
        metric_returns(ticker).shape[0]
        for ticker in universe_tickers
    ],
    "annualized_return": [
        annualized_return(metric_returns(ticker), TRADING_DAYS)
        for ticker in universe_tickers
    ],
    "annualized_volatility": [
        annualized_volatility(metric_returns(ticker), TRADING_DAYS)
        for ticker in universe_tickers
    ],
    "max_drawdown": [
        max_drawdown(metric_returns(ticker))
        for ticker in universe_tickers
    ]
})

asset_return_stats["sharpe_ratio"] = (
    (asset_return_stats["annualized_return"] - RISK_FREE_RATE)
    / asset_return_stats["annualized_volatility"]
)

asset_return_stats

In [ ]:
asset_universe_metrics = asset_universe.merge(
    asset_return_stats,
    on="yf_ticker",
    how="left"
)

asset_universe_metrics.sort_values(
    "sharpe_ratio",
    ascending=False
)

## 10. Exports

In [ ]:
asset_universe_overview.to_csv(
    DATA_PATH / "asset_universe_overview.csv",
    index=False
)

asset_universe_metrics.to_csv(
    DATA_PATH / "asset_universe_metrics.csv",
    index=False
)

print("Asset universe exports saved.")